# Base Model Evaluation (GPT-4.1-nano)

Evaluates the untuned GPT-4.1-nano on the FEVER **dev set** to establish a baseline.
Test set is held out for final comparison only.

In [ ]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = ""

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ""

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

# TODO: replace with actual deployment name once TA provisions gpt-4.1-nano
BASE_DEPLOYMENT = "gpt-4.1-nano"

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


In [21]:
import json
import time
import random
from collections import Counter, defaultdict

random.seed(42)

# stratified sample from dev set: 667 per class -> 2001 total
# test set is held out for final reporting
dev_data = []
with open("data/joined/fever_dev_joined.jsonl") as f:
    for line in f:
        dev_data.append(json.loads(line))

buckets = defaultdict(list)
for ex in dev_data:
    buckets[ex["label"]].append(ex)

eval_sample = []
for lbl, items in buckets.items():
    random.shuffle(items)
    eval_sample.extend(items[:667])
random.shuffle(eval_sample)

print(f"dev sample: {len(eval_sample)} examples")
print(Counter(ex["label"] for ex in eval_sample))

dev sample: 2001 examples
Counter({'SUPPORTED': 667, 'CONTRADICTED': 667, 'NOT MENTIONED': 667})


## Prompt Design

We try two prompts to document what works best:
- **Prompt A**, minimal: just ask for the label, no explanation required
- **Prompt B**, structured: ask for label + one-sentence justification (matches SFT/DPO format)
- **Prompt SFT**, exact training prompt used for SFT/DPO/RFT: used as the matched-prompt baseline for head-to-head comparison

In [26]:
# Prompt A: minimal, label only, no explanation
PROMPT_A_SYSTEM = """You are a fact-checking assistant. Given a passage and a claim, respond with only one word: SUPPORTED, CONTRADICTED, or NOT MENTIONED."""

# Prompt B: structured, matches the RLHF output format
PROMPT_B_SYSTEM = """You are a fact-checking assistant. Given a passage and a claim, respond with a JSON object:
{"label": "<SUPPORTED|CONTRADICTED|NOT MENTIONED>", "justification": "<one sentence citing the passage>"}"""

VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}

# 3-shot examples: one per class, kept simple and unambiguous
# injected as prior turns so the model sees all three label types before the real question
FEW_SHOT_A = [
    {"role": "user",      "content": "Passage: The Eiffel Tower is located in Paris, France.\n\nClaim: The Eiffel Tower is in Paris."},
    {"role": "assistant", "content": "SUPPORTED"},
    {"role": "user",      "content": "Passage: The Amazon River flows through Brazil.\n\nClaim: The Amazon River is in Europe."},
    {"role": "assistant", "content": "CONTRADICTED"},
    {"role": "user",      "content": "Passage: Mount Everest is the tallest mountain in the world.\n\nClaim: Mount Everest was first climbed in 1850."},
    {"role": "assistant", "content": "NOT MENTIONED"},
]

FEW_SHOT_B = [
    {"role": "user",      "content": "Passage: The Eiffel Tower is located in Paris, France.\n\nClaim: The Eiffel Tower is in Paris."},
    {"role": "assistant", "content": '{"label": "SUPPORTED", "justification": "The passage states the Eiffel Tower is located in Paris, France, directly supporting the claim."}'},
    {"role": "user",      "content": "Passage: The Amazon River flows through Brazil.\n\nClaim: The Amazon River is in Europe."},
    {"role": "assistant", "content": '{"label": "CONTRADICTED", "justification": "The passage states the Amazon River flows through Brazil, contradicting the claim that it is in Europe."}'},
    {"role": "user",      "content": "Passage: Mount Everest is the tallest mountain in the world.\n\nClaim: Mount Everest was first climbed in 1850."},
    {"role": "assistant", "content": '{"label": "NOT MENTIONED", "justification": "The passage only states that Mount Everest is the tallest mountain and does not mention when it was first climbed."}'},
]

import re

def extract_label(text):
    # 1. exact match (Prompt A)
    if text in VALID_LABELS:
        return text
    # 2. "LABEL: justification" format (SFT/DPO/RFT training format)
    for lbl in VALID_LABELS:
        if text.upper().startswith(lbl + ":"):
            return lbl
    # 3. JSON format (Prompt B)
    try:
        parsed = json.loads(text)
        label = parsed.get("label", "").strip()
        if label in VALID_LABELS:
            return label
    except Exception:
        pass
    # 4. last resort: first valid label as a whole word, longest match first
    for lbl in sorted(VALID_LABELS, key=len, reverse=True):
        if re.search(rf'\b{lbl}\b', text.upper()):
            return lbl
    return None

def predict(passage, claim, system_prompt, temperature=0.0, few_shot=None):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}"
    messages = [{"role": "system", "content": system_prompt}]
    if few_shot:
        messages.extend(few_shot)
    messages.append({"role": "user", "content": user_msg})

    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=BASE_DEPLOYMENT,
                messages=messages,
                temperature=temperature,
                max_tokens=150,
            )
            text = resp.choices[0].message.content.strip()
            label = extract_label(text)
            if label:
                return label, text
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None, None

## Hyperparameter Configurations

We run six configurations varying prompt and temperature to see what affects accuracy most.

In [27]:
# Matched Prompt: exact prompt used during SFT/DPO/RFT training, for matched-prompt baseline comparison
PROMPT_MATCHED_SYSTEM = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

# configs: vary prompt type, temperature, and few-shot
CONFIGS = [
    {"name": "A_temp0",       "system": PROMPT_A_SYSTEM,   "temperature": 0.0, "few_shot": None},
    {"name": "B_temp0",       "system": PROMPT_B_SYSTEM,   "temperature": 0.0, "few_shot": None},
    {"name": "B_temp0.3",     "system": PROMPT_B_SYSTEM,   "temperature": 0.3, "few_shot": None},
    {"name": "A_3shot_temp0", "system": PROMPT_A_SYSTEM,   "temperature": 0.0, "few_shot": FEW_SHOT_A},
    {"name": "B_3shot_temp0", "system": PROMPT_B_SYSTEM,   "temperature": 0.0, "few_shot": FEW_SHOT_B},
    {"name": "Matched_prompt",    "system": PROMPT_MATCHED_SYSTEM, "temperature": 0.0, "few_shot": None},
]

def run_eval(config, examples, out_path):
    # resume from partial run if output already exists
    done_ids = set()
    results = []
    if os.path.exists(out_path):
        with open(out_path) as f:
            for line in f:
                r = json.loads(line)
                done_ids.add(r["id"])
                results.append(r)
        print(f"resuming, {len(done_ids)} already done")

    skipped = 0
    with open(out_path, "a") as out_f:
        for i, ex in enumerate(examples):
            if ex["id"] in done_ids:
                continue
            pred_label, raw = predict(ex["passage"], ex["claim"],
                                      config["system"], config["temperature"],
                                      config.get("few_shot"))
            if pred_label is None:
                skipped += 1
                continue

            record = {
                "id":         ex["id"],
                "label":      ex["label"],
                "pred_label": pred_label,
                "raw":        raw,
            }
            out_f.write(json.dumps(record) + "\n")
            out_f.flush()
            results.append(record)

            if (i + 1) % 200 == 0:
                print(f"  [{i+1}/{len(examples)}] skipped={skipped}")

    print(f"done. skipped={skipped}")
    return results

In [31]:
# run all configs and collect results
all_results = {}
for cfg in CONFIGS:
    print(f"Config: {cfg['name']}")
    out_path = f"data/results/baseline_{cfg['name']}_dev.jsonl"
    results = run_eval(cfg, eval_sample, out_path)
    all_results[cfg["name"]] = results
    print()

Config: A_temp0
resuming, 1994 already done
  attempt 1 failed: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'medium'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  attempt 2 failed: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more a

## Results

In [38]:
def compute_metrics(results):
    labels = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]
    per_class = {lbl: {"correct": 0, "total": 0} for lbl in labels}
    confusion = {true: {pred: 0 for pred in labels} for true in labels}
    total_correct = 0

    for r in results:
        gt, pred = r["label"], r["pred_label"]
        per_class[gt]["total"] += 1
        confusion[gt][pred] += 1
        if pred == gt:
            per_class[gt]["correct"] += 1
            total_correct += 1

    macro_acc = sum(
        per_class[lbl]["correct"] / per_class[lbl]["total"]
        for lbl in labels if per_class[lbl]["total"] > 0
    ) / len(labels)
    overall_acc = total_correct / len(results)

    print(f"  overall accuracy: {overall_acc:.3f}  ({total_correct}/{len(results)})")
    print(f"  macro accuracy:   {macro_acc:.3f}")
    print()
    print("  Per-class accuracy:")
    for lbl in labels:
        c = per_class[lbl]
        acc = c["correct"] / c["total"] if c["total"] else 0
        print(f"    {lbl:20s}: {acc:.3f}  ({c['correct']}/{c['total']})")
    print()
    print("  Confusion matrix (rows=true, cols=pred):")
    col_w = 14
    header = " " * 22 + "".join(f"{lbl[:col_w]:>{col_w}}" for lbl in labels)
    print(header)
    for true_lbl in labels:
        row = f"  {true_lbl:20s}" + "".join(
            f"{confusion[true_lbl][pred_lbl]:>{col_w}}" for pred_lbl in labels
        )
        print(row)

    return macro_acc

# load results from disk so this cell is self-contained
all_results = {}
for cfg in CONFIGS:
    path = f"data/results/baseline_{cfg['name']}_dev.jsonl"
    if os.path.exists(path):
        records = []
        with open(path) as f:
            for line in f:
                if not line.strip(): continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
        all_results[cfg["name"]] = records
    else:
        print(f"missing: {path}")

print("Baseline Results\n")
best_cfg, best_acc = None, 0
for cfg_name, results in all_results.items():
    print(f"Config: {cfg_name}")
    acc = compute_metrics(results)
    if acc > best_acc:
        best_acc, best_cfg = acc, cfg_name
    print()

print(f"Best config: {best_cfg}  (macro accuracy={best_acc:.3f})")

Baseline Results

Config: A_temp0
  overall accuracy: 0.640  (640/1000)
  macro accuracy:   0.643

  Per-class accuracy:
    SUPPORTED           : 0.767  (250/326)
    CONTRADICTED        : 0.967  (324/335)
    NOT MENTIONED       : 0.195  (66/339)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      250            51            25
  CONTRADICTED                     4           324             7
  NOT MENTIONED                    0           273            66

Config: B_temp0
  overall accuracy: 0.823  (823/1000)
  macro accuracy:   0.821

  Per-class accuracy:
    SUPPORTED           : 0.623  (203/326)
    CONTRADICTED        : 0.869  (291/335)
    NOT MENTIONED       : 0.971  (329/339)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      203            17           106
  CONTRADICTED                     4    

## Final Test Evaluation (Matched Prompt)

Run once on the held-out test set using the Matched_prompt config.
This is the fair head-to-head baseline against SFT/DPO/RFT.
Do not re-run after inspecting results.

In [ ]:
random.seed(42)
test_data = []
with open("data/joined/fever_test_joined.jsonl") as f:
    for line in f:
        test_data.append(json.loads(line))

test_buckets = defaultdict(list)
for ex in test_data:
    test_buckets[ex["label"]].append(ex)

test_sample = []
for lbl, items in sorted(test_buckets.items()):
    random.shuffle(items)
    test_sample.extend(items[:667])
random.shuffle(test_sample)

print(f"test sample: {len(test_sample)} examples")
print(Counter(ex["label"] for ex in test_sample))

matched_cfg = {"name": "Matched_prompt", "system": PROMPT_MATCHED_SYSTEM, "temperature": 0.0, "few_shot": None}
test_results = run_eval(matched_cfg, test_sample, "data/results/baseline_Matched_prompt_test.jsonl")

In [39]:
# load test results from disk and report metrics
test_records = []
with open("data/results/baseline_Matched_prompt_test.jsonl") as f:
    for line in f:
        if not line.strip(): continue
        try:
            test_records.append(json.loads(line))
        except json.JSONDecodeError:
            pass

print(f"Final Test Result: Matched_prompt (baseline)  n={len(test_records)}")
compute_metrics(test_records)

Final Test Result: Matched_prompt (baseline)  n=1999
  overall accuracy: 0.642  (1283/1999)
  macro accuracy:   0.642

  Per-class accuracy:
    SUPPORTED           : 0.820  (547/667)
    CONTRADICTED        : 0.962  (640/665)
    NOT MENTIONED       : 0.144  (96/667)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      547            76            44
  CONTRADICTED                    11           640            14
  NOT MENTIONED                    0           571            96


0.6421413353473638